In [1]:
import json
import math
import time
from glob import glob
from pathlib import Path

import geopandas as gpd
import pandas as pd
import requests
import yaml
from shapely.geometry.base import BaseGeometry


def load_config(path="config.yaml"):
    with open(path, "r", encoding="utf-8") as f:
        return yaml.safe_load(f)


def ensure_dir(p: Path) -> Path:
    p.mkdir(parents=True, exist_ok=True)
    return p


def clamp_xpv(xpv_ha: float) -> float:
    if xpv_ha < 2 or xpv_ha > 100:
        raise ValueError(f"XPV requested_area_ha must be in [2, 100]. Got {xpv_ha}")
    return xpv_ha


def ha_to_m2(ha: float) -> float:
    return float(ha) * 10000.0


def panel_area_m2(cfg) -> float:
    return float(cfg["panel"]["length_m"]) * float(cfg["panel"]["width_m"])


def guess_nuts_layer(gpkg_path: str, preferred=None):
    layers = gpd.list_layers(gpkg_path)
    if preferred:
        return preferred
    return layers.iloc[0]["name"] if len(layers) else None


def county_name_from_row(row) -> str:
    for col in ["NAME_LATN", "NAME_ENGL", "NAME", "NUTS_NAME"]:
        if col in row and pd.notna(row[col]):
            return str(row[col])
    return str(row["NUTS_ID"])


def load_counties(cfg) -> gpd.GeoDataFrame:
    gpkg_path = cfg["project"]["nuts_gpkg_path"]
    layer = guess_nuts_layer(gpkg_path, cfg["project"].get("nuts_layer"))
    if not layer:
        raise RuntimeError(f"No layers found in {gpkg_path}")

    gdf = gpd.read_file(gpkg_path, layer=layer)

    if "CNTR_CODE" in gdf.columns:
        gdf = gdf[gdf["CNTR_CODE"] == cfg["project"]["country"]].copy()
    if "LEVL_CODE" in gdf.columns:
        gdf = gdf[gdf["LEVL_CODE"] == cfg["project"]["nuts_level"]].copy()

    if "NUTS_ID" not in gdf.columns:
        raise RuntimeError("Expected column 'NUTS_ID' in GeoPackage layer.")

    if gdf.crs is None:
        gdf = gdf.set_crs("EPSG:4326")

    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    return gdf


def find_substation_files(cfg):
    explicit = cfg["grid_proxy"].get("substations_geojson")
    if explicit and Path(explicit).exists():
        return [explicit]

    pattern = cfg["grid_proxy"].get("substations_glob")
    if not pattern:
        return []

    return sorted(glob(pattern))


def sanitize_points(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    if gdf.empty:
        return gdf
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    gdf = gdf[gdf.geometry.apply(lambda x: isinstance(x, BaseGeometry))].copy()
    gdf["geometry"] = gdf.geometry.apply(
        lambda geom: geom if geom.geom_type == "Point" else geom.representative_point()
    )
    gdf = gdf[gdf.geometry.notna() & ~gdf.geometry.is_empty].copy()
    return gdf


def load_substations(cfg) -> gpd.GeoDataFrame:
    files = find_substation_files(cfg)
    if not files:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    frames = []
    for fp in files:
        try:
            g = gpd.read_file(fp)
            if g.crs is None:
                g = g.set_crs("EPSG:4326")
            g = sanitize_points(g)
            frames.append(g[["geometry"]].copy())
        except Exception:
            continue

    if not frames:
        return gpd.GeoDataFrame(geometry=[], crs="EPSG:4326")

    out = pd.concat(frames, ignore_index=True)
    out = gpd.GeoDataFrame(out, geometry="geometry", crs=frames[0].crs)
    out = sanitize_points(out)
    return out


def load_cache(cache_path: Path) -> dict:
    if cache_path.exists():
        return json.loads(cache_path.read_text(encoding="utf-8"))
    return {}


def save_cache(cache_path: Path, cache: dict):
    cache_path.write_text(json.dumps(cache, indent=2), encoding="utf-8")


def pvgis_annual_kwh_per_kwp(lat: float, lon: float, cfg, session: requests.Session, cache: dict) -> float:
    """
    PVGIS PVcalc returns E_y (kWh/year) for 1 kWp system at a location.
    """
    key = f"{lat:.5f},{lon:.5f}"
    if key in cache:
        return float(cache[key])

    base = cfg["pv_model"]["api_base"].rstrip("/")
    url = f"{base}/PVcalc"

    params = {
        "lat": lat,
        "lon": lon,
        "peakpower": 1.0,
        "loss": cfg["pv_model"]["system"]["loss_percent"],
        "angle": cfg["pv_model"]["system"]["tilt_deg"],
        "aspect": cfg["pv_model"]["system"]["azimuth_deg"],
        "pvtechchoice": cfg["pv_model"]["system"]["pvtechchoice"],
        "mountingplace": cfg["pv_model"]["system"]["mountingplace"],
        "outputformat": "json",
    }

    r = session.get(url, params=params, timeout=60)
    r.raise_for_status()
    data = r.json()
    e_y = float(data["outputs"]["totals"]["fixed"]["E_y"])
    cache[key] = e_y
    return e_y


def brute_grid_metrics(rep_point, substation_points, radius_m: float):
    """
    Brute-force distance + count.
    With 42 counties and ~1660 points, this is fast and robust.
    """
    if not substation_points:
        return float("inf"), 0

    dists = [rep_point.distance(p) for p in substation_points]
    d_min = min(dists)
    cnt = sum(1 for d in dists if d <= radius_m)
    return d_min, cnt


def print_block(title: str, df: pd.DataFrame, cols: list[str]):
    print("\n" + "=" * 110)
    print(title)
    print("=" * 110)
    if df.empty:
        print("(none)")
        return
    print(df[cols].to_string(index=False))


def load_land_map(cfg) -> dict:
    """
    Option 1: load CORINE-derived available land from CSV.
    """
    csv_path = Path(cfg["land"]["available_land_csv"])
    if not csv_path.exists():
        raise FileNotFoundError(
            f"Missing land availability CSV: {csv_path}\n"
            f"Run: python build_available_land.py\n"
            f"Then rerun: python main.py"
        )

    land_df = pd.read_csv(csv_path)
    if "NUTS_ID" not in land_df.columns or "available_ha" not in land_df.columns:
        raise ValueError(f"{csv_path} must have columns: NUTS_ID, available_ha")

    return dict(zip(land_df["NUTS_ID"], land_df["available_ha"]))


def main():
    cfg = load_config("config.yaml")

    land_map = load_land_map(cfg)

    debug = bool(cfg.get("debug", False))

    cache_dir = ensure_dir(Path(cfg["run"]["cache_dir"]))
    cache_path = cache_dir / cfg["run"]["pvgis_cache_file"]

    xpv = clamp_xpv(float(cfg["xpv"]["requested_area_ha"]))

    counties = load_counties(cfg)
    substations = load_substations(cfg)

    proj_crs = cfg["project"]["crs_area_distance"]
    counties_proj = counties.to_crs(proj_crs)
    substations_proj = substations.to_crs(proj_crs)

    sub_pts = substations_proj.geometry.tolist()

    if debug:
        print(f"[DEBUG] counties: {len(counties_proj)} CRS={counties_proj.crs}")
        print(f"[DEBUG] substations: {len(substations_proj)} CRS={substations_proj.crs}")
        if len(substations_proj) > 0:
            print("[DEBUG] geometry types:", substations_proj.geometry.geom_type.value_counts().to_dict())
        print(f"[DEBUG] land_map entries: {len(land_map)}")

    grid_cfg = cfg["constraints"]["grid"]
    require_grid = bool(grid_cfg.get("require_grid", True))
    max_km = float(grid_cfg["nearest_substation_max_km"])
    radius_km = float(grid_cfg["radius_km"])
    k_min = int(grid_cfg["min_substations_within_radius"])
    radius_m = radius_km * 1000.0

    cache = load_cache(cache_path)
    session = requests.Session()
    sleep_s = float(cfg["run"].get("request_sleep_s", 0.0))

    results = []

    for _, row in counties_proj.iterrows():
        county_id = row["NUTS_ID"]
        county_name = county_name_from_row(row)

        available_ha = float(land_map.get(county_id, 0.0))
        land_ok = available_ha >= xpv

        rep = row.geometry.representative_point()

        rep_wgs = gpd.GeoSeries([rep], crs=counties_proj.crs).to_crs("EPSG:4326").iloc[0]
        lat, lon = float(rep_wgs.y), float(rep_wgs.x)

        on_fail = cfg["pv_model"].get("on_fail", "fallback")
        fallback = float(cfg["pv_model"].get("fallback_Ey_kWh_per_kWp", 1300))

        error = None
        try:
            Ey_kWh_per_kWp = pvgis_annual_kwh_per_kwp(lat, lon, cfg, session, cache)
        except Exception as e:
            if on_fail == "skip":
                Ey_kWh_per_kWp = float("nan")
                error = f"PVGIS failed: {e}"
            else:
                Ey_kWh_per_kWp = fallback
                error = f"PVGIS failed (fallback used): {e}"

        panel_kwp = float(cfg["panel"]["p_stc_kwp"])
        gcr = float(cfg["panel"]["ground_coverage_ratio"])
        a_panel = panel_area_m2(cfg)

        eff_ha = min(xpv, available_ha)
        n_panels = math.floor((ha_to_m2(eff_ha) * gcr) / a_panel) if eff_ha > 0 else 0

        annual_energy_kWh = Ey_kWh_per_kWp * panel_kwp * n_panels

        d_m, cnt = brute_grid_metrics(rep, sub_pts, radius_m)
        d_km = d_m / 1000.0 if math.isfinite(d_m) else float("inf")

        grid_ok = True
        if require_grid:
            grid_ok = (d_km <= max_km) and (cnt >= k_min)

        eligible = land_ok and grid_ok

        results.append({
            "county_id": county_id,
            "county_name": county_name,
            "available_ha": available_ha,
            "requested_xpv_ha": xpv,
            "effective_ha_used": eff_ha,
            "missing_ha_to_fit": max(xpv - available_ha, 0.0),
            "land_ok": land_ok,
            "grid_ok": grid_ok,
            "eligible": eligible,

            "lat": lat,
            "lon": lon,
            "pvgis_Ey_kWh_per_kWp": Ey_kWh_per_kWp,
            "panel_kWp": panel_kwp,
            "n_panels_used": int(n_panels),
            "annual_energy_kWh": float(annual_energy_kWh),

            "nearest_substation_km": d_km if math.isfinite(d_km) else None,
            "substations_within_radius": int(cnt),

            "error": error,
        })

        if sleep_s > 0:
            time.sleep(sleep_s)

    save_cache(cache_path, cache)

    df = pd.DataFrame(results)

    df_eligible = df[df["eligible"] == True].copy()

    df_eligible = df_eligible.sort_values(
        by=["annual_energy_kWh", "pvgis_Ey_kWh_per_kWp", "nearest_substation_km", "substations_within_radius"],
        ascending=[False, False, True, False],
        na_position="last",
    )

    df_nofit_land = df[df["available_ha"] < xpv].copy()
    df_nofit_land = df_nofit_land.sort_values(
        by=["annual_energy_kWh", "grid_ok", "pvgis_Ey_kWh_per_kWp", "nearest_substation_km"],
        ascending=[False, False, False, True],
        na_position="last",
    )

    out_csv = Path(cfg["run"]["output_csv"])
    out_json = Path(cfg["run"]["output_json"])
    ensure_dir(out_csv.parent)
    ensure_dir(out_json.parent)

    df.to_csv(out_csv, index=False, encoding="utf-8")
    df.to_json(out_json, orient="records", indent=2, force_ascii=False)

    top_n = int(cfg["run"]["top_n"])
    extra_nn = int(cfg["run"]["extra_nn"])

    err_rows = df["error"].notna().sum()
    if err_rows > 0:
        print(f"\n{err_rows}/{len(df)} rows had PVGIS errors (fallback may have been used).")
        print(df.loc[df["error"].notna(), ["county_id", "county_name", "error"]].head(3).to_string(index=False))

    if df_eligible.empty:
        print(f"\nNo counties satisfy BOTH hard constraints for XPV={xpv:.1f} ha:")
        print(f"   - land_ok: available_ha >= {xpv:.1f}")
        print(f"   - grid_ok: nearest_substation_km <= {max_km} AND substations within {radius_km}km >= {k_min}")
        print(f"\nShowing {extra_nn} best alternatives if you needed smaller land (available_ha < XPV):\n")

        show = df_nofit_land.head(extra_nn)
        print_block(
            f"Best {extra_nn} smaller-land options (ranked by total annual energy achievable)",
            show,
            cols=[
                "county_id", "county_name",
                "available_ha", "missing_ha_to_fit",
                "grid_ok", "nearest_substation_km", "substations_within_radius",
                "n_panels_used", "annual_energy_kWh"
            ],
        )
    else:
        top = df_eligible.head(top_n)
        print_block(
            f"Top {top_n} counties (HARD constraints satisfied) — ranked by TOTAL annual energy",
            top,
            cols=[
                "county_id", "county_name",
                "available_ha",
                "nearest_substation_km", "substations_within_radius",
                "n_panels_used", "annual_energy_kWh"
            ],
        )

        extra = df_nofit_land.head(extra_nn)
        print(f"\nNote: The counties below do NOT fit XPV={xpv:.1f} ha by land,")
        print("but are shown as strong candidates if you needed a smaller area. Each shows exact available land.\n")

        print_block(
            f"Extra {extra_nn} smaller-land options (available_ha < XPV) — ranked by achievable annual energy",
            extra,
            cols=[
                "county_id", "county_name",
                "available_ha", "missing_ha_to_fit",
                "grid_ok", "nearest_substation_km", "substations_within_radius",
                "n_panels_used", "annual_energy_kWh"
            ],
        )

    print("\nSaved outputs:")
    print(f" - {out_csv.resolve()}")
    print(f" - {out_json.resolve()}")
    print(f" - PVGIS cache: {cache_path.resolve()}")


if __name__ == "__main__":
    main()


Top 8 counties (HARD constraints satisfied) — ranked by TOTAL annual energy
county_id county_name  available_ha  nearest_substation_km  substations_within_radius  n_panels_used  annual_energy_kWh
    RO225      Tulcea      374446.0               4.396129                         22         122448       73024497.432
    RO223   Constanţa      556336.0               4.054993                         89         122448       72487807.848
    RO414         Olt      436624.0               3.735402                         15         122448       72103749.696
    RO411        Dolj      577548.0              10.567411                         15         122448       71938995.912
    RO313   Dâmboviţa      220525.0               2.036367                         12         122448       71666794.008
    RO321   Bucureşti        3322.0               0.885282                        213         122448       71492672.952
    RO221      Brăila      401472.0               5.919751                         